# Gerador de Dimensão: Produtos

**Objetivo:** criar a carga inicial da dimensão `dim_produtos`, contendo o catálogo fixo de 6 SKUs (3 produtos × 2 tamanhos), com nomes traduzidos para os 4 idiomas do projeto (português, espanhol argentino, espanhol mexicano, inglês) e preços em moeda local por país.

**Dependências:** `src/utils/scd2_handler.py` (classe `SCD2Handler`, para aplicar o controle de versionamento SCD Tipo 2).

**Widget:** `forcar_recriacao` — permite re-executar este notebook durante testes/desenvolvimento sem necessidade de alteração manual de código.

**Saída:** arquivo JSON gravado na Landing Zone (`/Volumes/poc_latam_food/landing/blob_simulado/dimensoes/produtos/`), simulando a origem externa dos dados antes da ingestão via Autoloader.

**Observação:** diferente do notebook de Representantes (que usará Faker para gerar nomes aleatórios), este notebook utiliza uma **lista fixa e manual** de produtos — pois o catálogo já é conhecido e definido, não sendo um dado que precisa ser gerado aleatoriamente.

In [0]:
# Instalações e configurações iniciais

import sys

# Adiciona a pasta 'src' ao path do Python, permitindo importar
# os módulos criados em utils/ como se fossem uma biblioteca comum.
sys.path.append("/Workspace/Users/bruno.quelestech@outlook.com/poc-lakehouse-food-latam/src")

from utils.scd2_handler import SCD2Handler

print("Dependências carregadas com sucesso.")

In [0]:
# Widget

dbutils.widgets.dropdown("forcar_recriacao", "false", ["true", "false"])

forcar_recriacao = dbutils.widgets.get("forcar_recriacao") == "true"

print(f"Forçar recriação: {forcar_recriacao}")

In [0]:
# Lista de produtos

produtos_cru = [
    # (produto_id, nome_interno, nome_brasil, nome_argentina, nome_mexico, nome_ingles, tamanho, preco_brl, preco_ars, preco_mxn)
    ("PROD001", "MAIONESE", "Maionese", "Mayonesa", "Mayonesa", "Mayonnaise", "1kg", "12,90", "2850,00", "45,00"),
    ("PROD002", "MAIONESE", "Maionese", "Mayonesa", "Mayonesa", "Mayonnaise", "5kg", "52,90", "11500,00", "185,00"),
    ("PROD003", "MOSTARDA", "Mostarda", "Mostaza", "Mostaza", "Mustard", "1kg", "9,90", "2200,00", "35,00"),
    ("PROD004", "MOSTARDA", "Mostarda", "Mostaza", "Mostaza", "Mustard", "5kg", "41,90", "9200,00", "145,00"),
    ("PROD005", "KETCHUP", "Ketchup", "Ketchup", "Catsup", "Ketchup", "1kg", "11,50", "2500,00", "40,00"),
    ("PROD006", "KETCHUP", "Ketchup", "Ketchup", "Catsup", "Ketchup", "5kg", "47,90", "10400,00", "165,00"),
]

colunas_produtos = [
    "produto_id", "nome_interno", "nome_brasil", "nome_argentina",
    "nome_mexico", "nome_ingles", "tamanho",
    "preco_brasil_brl", "preco_argentina_ars", "preco_mexico_mxn"
]

print(f"Total de produtos definidos: {len(produtos_cru)}")

In [0]:
# Transformação em dataframe spark

df_produtos_cru = spark.createDataFrame(produtos_cru, colunas_produtos)

df_produtos_cru.display()

In [0]:
# Aplicar o SCD2Handler

scd2 = SCD2Handler()
df_produtos_scd2 = scd2.iniciar_controle_scd2(df_produtos_cru)

df_produtos_scd2.display()

In [0]:
# Gravar o resultado como JSON na Landing Zone

caminho_landing = "/Volumes/poc_latam_food/landing/blob_simulado/dimensoes/produtos"

# Verifica se já existem arquivos naquele caminho
arquivos_existentes = []
try:
    arquivos_existentes = dbutils.fs.ls(caminho_landing)
except Exception:
    # Pasta ainda não existe, é a primeira execução
    pass

if len(arquivos_existentes) > 0 and not forcar_recriacao:
    print("Já existem arquivos na Landing Zone de produtos. "
          "Nenhuma ação foi tomada (use o widget 'forcar_recriacao' = true para sobrescrever).")
else:
    (
        df_produtos_scd2
        .coalesce(1)
        .write
        .mode("overwrite")
        .json(caminho_landing)
    )
    print(f"Arquivo gravado com sucesso em: {caminho_landing}")